# Synthetic Data Generation for Recipe RFT/SFT/DPO

**CIS 5270 Course Project — Team: Akshat, Rashi, Ritika**

This notebook implements the synthetic data generation pipeline described in the project proposal.

### Pipeline (from the proposal and pseudocode)

```
L = []
N = 1000 - 5000
while len(L) <= N:
    (x)    1st LLM call: Generate random ingredients list (seeded with cuisine/meal-type)
    (y)    2nd LLM call: Using list, generate a constraint-satisfying student-friendly recipe
    (y')   3rd LLM call: Generate a negative recipe (constraint-violating) for DPO
    4th LLM call: LLM-as-judge to verify (x, y) sanity check
    if sanity_check_passed:
        L.append((x, y, y'))
```

### Outputs

The pipeline produces a single pool of `(x, y, y')` triples, which are then split into the JSONL files the Azure fine-tuning jobs expect:

- `sft_train.jsonl` / `sft_val.jsonl` — messages format, `(x, y)` only
- `dpo_train.jsonl` / `dpo_val.jsonl` — preference format, `(x, y, y')`

### Design choices

- **Ingredient generation**: hybrid seeding. We sample a `(cuisine, meal_type, difficulty_hint)` tuple and ask GPT-5-mini to produce a realistic ingredient list. This gives us diversity without hardcoding thousands of lists.
- **Preferred vs rejected**: generated in **separate calls** with different system prompts — the preferred prompt enforces constraints, the rejected prompt is explicitly instructed to violate one or more (e.g., ignore some ingredients, be long-winded, skip structure). Cleaner contrast for DPO than asking one call for both.
- **Sanity check**: the 4th call (LLM-as-judge) returns a structured JSON verdict. Only samples that pass are appended.
- **Robustness**: every LLM call is wrapped in retry-with-backoff, all outputs are validated as JSON before use, and the pipeline checkpoints to disk every N samples so a crash doesn't lose progress.
- **Concurrency**: uses `ThreadPoolExecutor` so the four sequential LLM calls per sample run in parallel across many samples. This is the dominant cost of the pipeline.

## 1. Setup and Azure client

Same auth block as the starter notebook — update `RESOURCE_GROUP` and `OPENAI_API_KEY` with your team's Gradescope credentials.

In [ ]:
%pip install -q openai azure-identity python-dotenv tqdm

In [ ]:
%pip install -q \
  "azure-ai-projects>=2.0.0b1" \
  openai \
  azure-identity \
  azure-mgmt-cognitiveservices \
  "azure-ai-evaluation>=1.13.0" \
  python-dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.7/91.7 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.5/59.5 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.6/48.6 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.3/274.3 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.1/192.1 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 290.1/290.1 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 218.3/218.3 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 431.5/431.5 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.5/121.5 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.4/85.4 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import os
import json
import time
import random
import hashlib
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass, asdict
from typing import Optional

from openai import AzureOpenAI
from tqdm.auto import tqdm

In [ ]:
import os
from openai import AzureOpenAI
from azure.identity import DefaultAzureCredential
from azure.mgmt.cognitiveservices import CognitiveServicesManagementClient
from azure.mgmt.cognitiveservices.models import Deployment, DeploymentProperties, DeploymentModel, Sku

In [ ]:
# TODO: Update with your team's credentials from Gradescope
RESOURCE_GROUP = 'cis-5270-team-14'
OPENAI_API_KEY = ''
OPENAI_ENDPOINT = f"https://{RESOURCE_GROUP}.openai.azure.com"
SUBSCRIPTION_ID = ''

openai_client = AzureOpenAI(
    api_key=OPENAI_API_KEY,
    azure_endpoint=OPENAI_ENDPOINT,
    api_version="2025-04-01-preview",
)

GENERATOR_MODEL = "gpt-4.1-mini"
JUDGE_MODEL = "gpt-4.1-mini"

print("Connected to Azure OpenAI")

Connected to Azure OpenAI


In [ ]:
# Azure resource targeting
os.environ["AZURE_SUBSCRIPTION_ID"] = SUBSCRIPTION_ID
os.environ["AZURE_RESOURCE_GROUP"] = 'CIS-5270'
os.environ["AZURE_AOAI_ACCOUNT"] = RESOURCE_GROUP
os.environ["AZURE_OPENAI_API_KEY"] = OPENAI_API_KEY
os.environ["AZURE_OPENAI_ENDPOINT"] = OPENAI_ENDPOINT
CREDENTIAL = DefaultAzureCredential()

## 2. Pipeline configuration

In [ ]:
@dataclass
class Config:
    # Target dataset size.
    n_samples: int = 2000

    # Exact target mix for the constrained dataset.
    category_targets: dict = None

    # Train/val split.
    val_fraction: float = 0.20

    # Concurrency — how many samples to generate in parallel.
    max_workers: int = 4

    # Retries per LLM call.
    max_retries: int = 4

    # Recipe constraints (must match the reward function used at train time).
    min_steps: int = 3
    max_steps: int = 7
    max_cook_minutes: int = 15

    # Checkpointing — flush the pool to disk every N accepted samples.
    checkpoint_every: int = 50

    # Output directory.
    out_dir: str = "data_constrained"

    # Reproducibility.
    seed: int = 42

    def __post_init__(self):
        if self.category_targets is None:
            self.category_targets = {
                "cuisine_style": 400,
                "weird_combo": 400,
                "adversarial": 400,
                "exclude_ingredient": 400,
                "low_calorie": 200,
                "taste_profile": 200,
            }
        self.n_samples = sum(self.category_targets.values())

CFG = Config()
Path(CFG.out_dir).mkdir(exist_ok=True)
random.seed(CFG.seed)

print(f"Target: {CFG.n_samples} samples -> {CFG.out_dir}/")
print("Category targets:")
for k, v in CFG.category_targets.items():
    print(f"  {k:20s} {v}")


Target: 2000 samples -> data_constrained/
Category targets:
  cuisine_style        400
  weird_combo          400
  adversarial          400
  exclude_ingredient   400
  low_calorie          200
  taste_profile        200


## 3. Seed pantry for hybrid ingredient generation

We don't hardcode ingredient lists — we hardcode *seeds* (cuisine, meal type, a hint or two) and let GPT-5-mini invent a plausible ingredient list. This gets us diversity cheaply without drifting into repetitive outputs the way free-form generation does.

In [ ]:
CUISINES = [
    "Italian", "Mexican", "Indian", "Chinese", "Thai", "American",
    "Korean", "Japanese", "French", "Spanish", "British",
]

TASTE_PROFILES = ["sweet", "sweet_savory", "spicy", "tangy"]

ADVERSARIAL_RESTRICTIONS = ["vegan", "vegetarian", "jain_friendly", "dairy_free", "gluten_free"]

EXCLUDE_TARGETS = [
    "cheese", "egg", "avocado", "soy sauce", "tuna", "yogurt", "chicken",
    "shrimp", "beans", "salmon", "mushroom", "garlic", "tomato", "milk",
]

def sample_seed(rng: random.Random, constraint_category: str) -> dict:
    """Sample metadata for the first LLM call.

    The category is fixed by the run scheduler so we can hit the exact desired
    dataset mix: 400 cuisine, 400 weird, 400 adversarial, 400 exclude,
    200 low-calorie, and 200 taste-profile examples.
    """
    seed = {
        "constraint_category": constraint_category,
        "n_ingredients": rng.randint(6, 8),
    }

    if constraint_category == "cuisine_style":
        cuisine = rng.choice(CUISINES)
        seed.update({
            "cuisine": cuisine,
            "constraint": f"{cuisine.lower()}_style",
            "user_constraint": f"{cuisine}-style",
        })

    elif constraint_category == "weird_combo":
        seed.update({
            "constraint": "weird_combination",
            "user_constraint": "using an unusual but coherent ingredient combination",
        })

    elif constraint_category == "adversarial":
        restriction = rng.choice(ADVERSARIAL_RESTRICTIONS)
        seed.update({
            "restriction": restriction,
            "constraint": f"{restriction}_adversarial",
            "user_constraint": restriction.replace("_", "-"),
        })

    elif constraint_category == "exclude_ingredient":
        excluded = rng.choice(EXCLUDE_TARGETS)
        seed.update({
            "excluded_ingredient": excluded,
            "constraint": f"exclude_{excluded.replace(' ', '_')}",
            "user_constraint": f"Do not include {excluded}",
        })

    elif constraint_category == "low_calorie":
        seed.update({
            "constraint": "low_calorie",
            "user_constraint": "low-calorie",
        })

    elif constraint_category == "taste_profile":
        profile = rng.choice(TASTE_PROFILES)
        seed.update({
            "taste_profile": profile,
            "constraint": f"{profile}_style",
            "user_constraint": f"{profile.replace('_', '-')}-style",
        })

    else:
        raise ValueError(f"Unknown constraint category: {constraint_category}")

    return seed

# Preview
_preview_rng = random.Random(0)
for cat in CFG.category_targets:
    print(sample_seed(_preview_rng, cat))


{'constraint_category': 'cuisine_style', 'n_ingredients': 7, 'cuisine': 'Korean', 'constraint': 'korean_style', 'user_constraint': 'Korean-style'}
{'constraint_category': 'weird_combo', 'n_ingredients': 6, 'constraint': 'weird_combination', 'user_constraint': 'using an unusual but coherent ingredient combination'}
{'constraint_category': 'adversarial', 'n_ingredients': 7, 'restriction': 'gluten_free', 'constraint': 'gluten_free_adversarial', 'user_constraint': 'gluten-free'}
{'constraint_category': 'exclude_ingredient', 'n_ingredients': 7, 'excluded_ingredient': 'chicken', 'constraint': 'exclude_chicken', 'user_constraint': 'Do not include chicken'}
{'constraint_category': 'low_calorie', 'n_ingredients': 7, 'constraint': 'low_calorie', 'user_constraint': 'low-calorie'}
{'constraint_category': 'taste_profile', 'n_ingredients': 7, 'taste_profile': 'spicy', 'constraint': 'spicy_style', 'user_constraint': 'spicy-style'}


## 4. Robust LLM call helper

Every call in the pipeline needs retries and JSON validation. Wrap it once.

In [ ]:
def llm_json(
    system: str,
    user: str,
    model: str = GENERATOR_MODEL,
    max_retries: int = CFG.max_retries,
) -> Optional[dict]:
    """Call the LLM, expect a JSON object back, retry on errors/bad JSON."""
    for attempt in range(max_retries):
        try:
            resp = openai_client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": system},
                    {"role": "user", "content": user},
                ],
                response_format={"type": "json_object"},
            )
            content = resp.choices[0].message.content
            return json.loads(content)
        except json.JSONDecodeError as e:
            # Bad JSON - retry, the model sometimes wraps in code fences.
            if attempt == max_retries - 1:
                print(f"[warn] JSON parse failed after {max_retries} tries: {e}")
                return None
        except Exception as e:
            # Rate limit, transient network, etc. Exponential backoff.
            wait = (2 ** attempt) + random.random()
            if attempt == max_retries - 1:
                print(f"[warn] LLM call failed after {max_retries} tries: {e}")
                return None
            time.sleep(wait)
    return None

## 5. The four LLM calls

### Call 1 — Generate ingredient list `x`

Given a seed, produce a realistic ingredient list. We return structured JSON so downstream calls and the reward function can parse it reliably.

In [ ]:
INGREDIENTS_SYSTEM = """You generate stress-test recipe tasks for college-student recipe generation.

Your job is to create a realistic but challenging ingredient list and the user-facing constraint text.

Rules:
- Return ONLY a JSON object with keys:
  "ingredients" (list of strings),
  "context" (short string),
  "constraint_category" (string),
  "constraint" (string),
  "user_constraint" (string),
  and optionally "excluded_ingredient" for exclude_ingredient tasks.
- Each ingredient is a plain, lowercase name. No quantities.
- Use 6-8 ingredients.
- The ingredient list should be challenging but still usable for a quick student recipe.
- Do NOT include water, salt, pepper, or cooking oil in the list.

Category-specific rules:
- cuisine_style: choose ingredients where some fit the cuisine and some do not; avoid making the list too obviously cuisine-specific.
- weird_combo: mix unusual sweet/savory/dairy/protein ingredients, but keep it possible to make a coherent dish.
- adversarial: include at least one ingredient that conflicts with the requested restriction (e.g., vegan with chicken; gluten-free with wheat pasta), plus enough allowed ingredients to make a valid recipe without using the conflicting item.
- exclude_ingredient: include the excluded ingredient in the list, but also include enough alternatives so a recipe can avoid it.
- low_calorie: include both lower-calorie and higher-calorie variants/options in the same list, without making the contrast too obvious.
- taste_profile: include ingredients that can support the requested taste profile, plus a few neutral ingredients.
"""

def generate_ingredients(seed: dict) -> Optional[dict]:
    user = f"""Generate one recipe task for this scenario:
- constraint_category: {seed['constraint_category']}
- constraint: {seed['constraint']}
- user_constraint: {seed['user_constraint']}
- number of ingredients: {seed['n_ingredients']}

Additional seed metadata:
{json.dumps(seed, indent=2)}

Return JSON with this shape:
{{
  "ingredients": ["...", "..."],
  "context": "...",
  "constraint_category": "{seed['constraint_category']}",
  "constraint": "{seed['constraint']}",
  "user_constraint": "{seed['user_constraint']}"
}}"""
    result = llm_json(INGREDIENTS_SYSTEM, user)
    if result is None or "ingredients" not in result:
        return None

    ings = result["ingredients"]
    if not isinstance(ings, list) or len(ings) < 6 or len(ings) > 8:
        return None

    result["ingredients"] = [str(i).strip().lower() for i in ings if i]
    result["seed"] = seed

    # Trust the scheduler's metadata over the model's metadata.
    result["constraint_category"] = seed["constraint_category"]
    result["constraint"] = seed["constraint"]
    result["user_constraint"] = seed["user_constraint"]
    if "excluded_ingredient" in seed:
        result["excluded_ingredient"] = seed["excluded_ingredient"]

    return result


### Call 2 — Generate preferred recipe `y`

Given the ingredient list, produce a recipe that *satisfies* all the constraints: uses the provided ingredients, is quick, has clear step-by-step structure, and is student-friendly. This is what SFT will teach the student model to imitate.

In [ ]:
PREFERRED_SYSTEM = f"""You are a recipe writer for busy college students.

You will be given a list of ingredients and an additional user constraint. Generate a recipe that strictly satisfies ALL of these constraints:

1. **Ingredient discipline**: Use ONLY ingredients from the provided list, plus water, salt, pepper, and cooking oil. Do not introduce unrelated ingredients. You do NOT need to use every listed ingredient.
2. **Additional constraint following**: Follow the provided user constraint exactly. Examples: cuisine-style, weird-but-coherent combination, dietary restriction, exclude ingredient, low-calorie, or taste profile.
3. **Speed**: Total time (prep + cook) must be {CFG.max_cook_minutes} minutes or less.
4. **Simplicity**: Between {CFG.min_steps} and {CFG.max_steps} steps. No step should require specialty equipment.
5. **Structure**: Each step is a single clear imperative sentence.
6. **Student-friendly**: Assume a tiny kitchen (stovetop, microwave, basic knife). No unusual techniques.

Return ONLY a JSON object with this exact shape:
{{
  "title": "...",
  "ingredients": ["...", "...", ....],
  "total_time_minutes": <int>,
  "steps": ["...", "...", ...]
}}
"""

def generate_preferred_recipe(ingredients: list[str], context: str, constraint: str, user_constraint: str) -> Optional[dict]:
    user = f"""Ingredients available: {', '.join(ingredients)}
Context: {context}
Additional user constraint: {user_constraint}
Constraint label for judge: {constraint}

Write the recipe as JSON."""
    result = llm_json(PREFERRED_SYSTEM, user)
    if result is None:
        return None
    if not all(k in result for k in ("title", "total_time_minutes", "steps")):
        return None
    if not isinstance(result["steps"], list) or len(result["steps"]) == 0:
        return None
    return result


### Call 3 — Generate rejected recipe `y'`

For DPO we need a recipe that's *plausibly tempting but worse*. We pick a random failure mode and instruct the LLM to commit that specific violation. This produces sharper preference signal than asking for "a bad recipe" (which gives us obviously broken outputs that DPO learns little from).

In [ ]:
REJECTION_MODES = [
    {
        "name": "extra_ingredients",
        "instruction": "Add 2-3 ingredients that were NOT in the provided list. Otherwise the recipe should look reasonable.",
    },
    {
        "name": "too_long",
        "instruction": f"Make the recipe take much longer than {CFG.max_cook_minutes + 15} minutes (e.g., 30-60 minutes of cooking, marinating, or resting).",
    },
    {
        "name": "too_many_steps",
        "instruction": f"Write {CFG.max_steps + 5} or more steps. Be overly detailed and break simple actions into multiple micro-steps.",
    },
    {
        "name": "unsafe_equipment",
        "instruction": "Require specialty equipment a student wouldn't have.",
    },
    {
        "name": "constraint_violation",
        "instruction": "Violate the additional user constraint while otherwise making the recipe look reasonable. For example, ignore the requested style, use an excluded ingredient, choose high-calorie options for a low-calorie task, or use a forbidden/conflicting dietary ingredient.",
    },
]

REJECTED_SYSTEM_TEMPLATE = """You are writing an intentionally sub-optimal recipe for a preference-learning dataset.

You will be given an ingredient list and an additional user constraint. The recipe you produce MUST violate this specific constraint:

>> {mode_instruction}

Do NOT announce that the recipe is bad. Write it as if it were a real recipe. It should look reasonable on the surface.
The other constraints may be satisfied or not — only the one above is required to be violated.

Return ONLY a JSON object with this exact shape:
{{
  "title": "...",
  "ingredients": ["...", "...", ...],
  "total_time_minutes": <int>,
  "steps": ["...", "...", ...]
}}
"""

def generate_rejected_recipe(ingredients: list[str], context: str, constraint: str, user_constraint: str, rng: random.Random) -> Optional[dict]:
    mode = rng.choice(REJECTION_MODES)
    system = REJECTED_SYSTEM_TEMPLATE.format(mode_instruction=mode["instruction"])
    user = f"""Ingredients available: {', '.join(ingredients)}
Context: {context}
Additional user constraint: {user_constraint}
Constraint label: {constraint}

Write the recipe as JSON. Remember to violate the specified constraint."""
    result = llm_json(system, user)
    if result is None:
        return None
    if not all(k in result for k in ("title", "total_time_minutes", "steps")):
        return None
    result["_rejection_mode"] = mode["name"]
    return result


### Call 4 — LLM-as-judge sanity check

The judge gets the ingredient list and the preferred recipe, and checks:
- Does the recipe use the provided ingredients (and not invent new ones)?
- Is it actually student-friendly (time, simplicity, structure)?
- Is the JSON well-formed semantically (e.g., `total_time_minutes` is plausible)?

We drop any sample the judge rejects. This is the step that keeps the dataset clean — the proposal's reward function depends on `(x, y)` being a valid positive example.

In [ ]:
JUDGE_SYSTEM = f"""You are a strict evaluator verifying a recipe dataset.

You will see (1) an ingredient list, (2) an additional user constraint, and (3) a recipe. Check ALL of these:

1. ingredient_compliance: The recipe uses only ingredients from the list, plus water/salt/pepper/oil/butter. It does not introduce unrelated ingredients. The recipe does not need to use every listed ingredient.
2. constraint_following: The recipe satisfies the additional user constraint. Examples: matches requested cuisine/taste style, handles weird ingredients coherently, respects dietary/adversarial restrictions, avoids excluded ingredients, and favors lower-calorie options when asked.
3. time_ok: total_time_minutes <= {CFG.max_cook_minutes} AND is plausible given the steps.
4. steps_ok: between {CFG.min_steps} and {CFG.max_steps} steps, each a clear single instruction.
5. student_friendly: no specialty equipment, realistic for a dorm kitchen.
6. coherent: the recipe actually makes a reasonable dish from these ingredients (not random word salad).

Return ONLY JSON:
{{
  "ingredient_compliance": <bool>,
  "constraint_following": <bool>,
  "time_ok": <bool>,
  "steps_ok": <bool>,
  "student_friendly": <bool>,
  "coherent": <bool>,
  "pass": <bool>,
  "reason": "<short explanation if pass is false, else empty>"
}}

Set "pass" to true only if ALL six checks are true.
"""

def judge_sample(ingredients: list[str], recipe: dict, constraint: str, user_constraint: str) -> Optional[dict]:
    user = f"""Ingredient list: {', '.join(ingredients)}
Additional user constraint: {user_constraint}
Constraint label: {constraint}

Recipe:
{json.dumps(recipe, indent=2)}

Evaluate and return JSON."""
    result = llm_json(JUDGE_SYSTEM, user, model=JUDGE_MODEL)
    if result is None or "pass" not in result:
        return None
    return result


## 6. Per-sample pipeline (the four calls glued together)

One invocation of this function = one candidate sample (which may or may not pass the sanity check).

In [ ]:
def generate_one_sample(sample_id: int, constraint_category: str, rng: random.Random) -> Optional[dict]:
    """Run the 4-call pipeline for one sample. Returns the sample dict or None if rejected."""
    # Call 1: constrained ingredients/task.
    seed = sample_seed(rng, constraint_category)
    ing_result = generate_ingredients(seed)
    if ing_result is None:
        return None

    ingredients = ing_result["ingredients"]
    context = ing_result["context"]
    constraint_category = ing_result["constraint_category"]
    constraint = ing_result["constraint"]
    user_constraint = ing_result["user_constraint"]

    # Call 2: preferred recipe.
    preferred = generate_preferred_recipe(ingredients, context, constraint, user_constraint)
    if preferred is None:
        return None

    # Call 3: rejected recipe (runs independently of call 2).
    rejected = generate_rejected_recipe(ingredients, context, constraint, user_constraint, rng)
    if rejected is None:
        return None

    # Call 4: judge the preferred recipe.
    verdict = judge_sample(ingredients, preferred, constraint, user_constraint)
    if verdict is None or not verdict.get("pass", False):
        return None

    return {
        "id": sample_id,
        "seed": seed,
        "ingredients": ingredients,
        "context": context,
        "constraint_category": constraint_category,
        "constraint": constraint,
        "user_constraint": user_constraint,
        "preferred": preferred,
        "rejected": rejected,
        "verdict": verdict,
    }


In [ ]:
import time
import threading

class RateLimiter:
    def __init__(self, rate: int):
        self.rate = rate  # requests per second
        self.tokens = rate
        self.lock = threading.Lock()
        self.last = time.time()

    def acquire(self):
        while True:
            with self.lock:
                now = time.time()
                elapsed = now - self.last

                # Refill tokens
                self.tokens += elapsed * self.rate
                if self.tokens > self.rate:
                    self.tokens = self.rate

                self.last = now

                if self.tokens >= 1:
                    self.tokens -= 1
                    return

            time.sleep(0.01)
rate_limiter = RateLimiter(rate=20)

## 7. Run the pipeline

Parallel across samples. Each worker runs the 4 sequential calls for one sample. We keep submitting until we have `N` accepted samples — rejected samples are retried with a fresh seed, so the `i += 1` in the pseudocode only advances on a successful sanity check.

In [ ]:
def checkpoint(pool: list[dict], path: str):
    """Flush the current pool to a single JSONL file so we don't lose progress."""
    with open(path, "w") as f:
        for s in pool:
            f.write(json.dumps(s) + "\n")

def load_checkpoint(path: str) -> list[dict]:
    if not Path(path).exists():
        return []
    with open(path) as f:
        return [json.loads(line) for line in f]

def category_counts(pool: list[dict]) -> dict:
    counts = {k: 0 for k in CFG.category_targets}
    for s in pool:
        cat = s.get("constraint_category")
        if cat in counts:
            counts[cat] += 1
    return counts

def choose_category_for_submission(rng: random.Random, counts: dict, targets: dict) -> Optional[str]:
    remaining = {cat: targets[cat] - counts.get(cat, 0) for cat in targets}
    remaining = {cat: rem for cat, rem in remaining.items() if rem > 0}
    if not remaining:
        return None
    # Weighted by remaining quota so categories finish close to target.
    cats = list(remaining.keys())
    weights = list(remaining.values())
    return rng.choices(cats, weights=weights, k=1)[0]

def run_pipeline(cfg: Config) -> list[dict]:
    ckpt_path = f"{cfg.out_dir}/pool.jsonl"
    pool = load_checkpoint(ckpt_path)
    if pool:
        print(f"Resuming from checkpoint: {len(pool)} samples already generated.")

    master_rng = random.Random(cfg.seed)
    n_submitted = len(pool)
    n_rejected = 0
    n_over_quota = 0

    counts = category_counts(pool)
    print("Starting category counts:", counts)

    pbar = tqdm(total=cfg.n_samples, initial=len(pool), desc="accepted")

    with ThreadPoolExecutor(max_workers=cfg.max_workers) as ex:
        futures = {}

        def submit_one():
            nonlocal n_submitted
            cat = choose_category_for_submission(master_rng, counts, cfg.category_targets)
            if cat is None:
                return False
            worker_rng = random.Random(master_rng.random())
            rate_limiter.acquire()
            fut = ex.submit(generate_one_sample, n_submitted, cat, worker_rng)
            futures[fut] = cat
            n_submitted += 1
            return True

        for _ in range(min(cfg.max_workers * 2, cfg.n_samples - len(pool))):
            submit_one()


        while len(pool) < cfg.n_samples and futures:
            done = next(as_completed(futures))
            submitted_cat = futures.pop(done)
            try:
                sample = done.result()
            except Exception as e:
                print(f"[error] worker raised: {e}")
                sample = None

            if sample is not None:
                cat = sample.get("constraint_category", submitted_cat)
                # Enforce exact quotas even with concurrent workers.
                if counts.get(cat, 0) < cfg.category_targets[cat]:
                    pool.append(sample)
                    counts[cat] += 1
                    pbar.update(1)
                    pbar.set_postfix(
                        rejected=n_rejected,
                        over_quota=n_over_quota,
                        accept_rate=f"{len(pool)/(len(pool)+n_rejected+n_over_quota):.2f}",
                    )
                    if len(pool) % cfg.checkpoint_every == 0:
                        checkpoint(pool, ckpt_path)
                else:
                    n_over_quota += 1
            else:
                n_rejected += 1

            while len(pool) + len(futures) < cfg.n_samples:
                if not submit_one():
                    break

    pbar.close()
    checkpoint(pool, ckpt_path)
    print(f"\nDone. Accepted: {len(pool)}  Rejected: {n_rejected}  Over-quota discarded: {n_over_quota}")
    print("Final category counts:")
    for cat, target in cfg.category_targets.items():
        print(f"  {cat:20s} {counts.get(cat, 0):4d} / {target}")
    return pool


In [ ]:
pool = run_pipeline(CFG)

Starting category counts: {'cuisine_style': 0, 'weird_combo': 0, 'adversarial': 0, 'exclude_ingredient': 0, 'low_calorie': 0, 'taste_profile': 0}


[warn] LLM call failed after 4 tries: Error code: 429 - {'error': {'message': 'Too Many Requests', 'type': 'too_many_requests', 'param': None, 'code': 'too_many_requests'}}
[warn] LLM call failed after 4 tries: Error code: 429 - {'error': {'message': 'Too Many Requests', 'type': 'too_many_requests', 'param': None, 'code': 'too_many_requests'}}
[warn] LLM call failed after 4 tries: Error code: 429 - {'error': {'message': 'Too Many Requests', 'type': 'too_many_requests', 'param': None, 'code': 'too_many_requests'}}
[warn] LLM call failed after 4 tries: Error code: 429 - {'error': {'message': 'Too Many Requests', 'type': 'too_many_requests', 'param': None, 'code': 'too_many_requests'}}
[warn] LLM call failed after 4 tries: Error code: 429 - {'error': {'message': 'Too Many Requests', 'type': 'too_many_requests', 'param': None, 'code': 'too_many_requests'}}
[warn] LLM call failed after 4 tries: Error code: 429 - {'error': {'message': 'Too Many Requests', 'type': 'too_many_requests', 'param'

## 8. Quick sanity check on the pool

Before we split into SFT/DPO/RFT files, spot-check distribution and content.

In [ ]:
from collections import Counter

print(f"Pool size: {len(pool)}")

print(f"\nConstraint-category distribution:")
for cat, count in Counter(s.get("constraint_category", "?") for s in pool).most_common():
    print(f"  {cat:20s} {count}")

print(f"\nConstraint-label distribution:")
for label, count in Counter(s.get("constraint", "?") for s in pool).most_common(20):
    print(f"  {label:25s} {count}")

print(f"\nRejection-mode distribution (in y'):")
for mode, count in Counter(s['rejected'].get('_rejection_mode', '?') for s in pool).most_common():
    print(f"  {mode:20s} {count}")

print(f"\nRecipe length (preferred):")
step_counts = [len(s['preferred']['steps']) for s in pool]
print(f"  mean steps: {sum(step_counts)/len(step_counts):.2f}")
print(f"  min/max:    {min(step_counts)} / {max(step_counts)}")

print(f"\nSample example:")
print(json.dumps(pool[0], indent=2)[:2000])


Pool size: 2000

Constraint-category distribution:
  exclude_ingredient   400
  adversarial          400
  weird_combo          400
  cuisine_style        400
  low_calorie          200
  taste_profile        200

Constraint-label distribution:
  weird_combination         400
  low_calorie               200
  vegetarian_adversarial    107
  vegan_adversarial         96
  dairy_free_adversarial    87
  gluten_free_adversarial   86
  spicy_style               61
  tangy_style               54
  sweet_savory_style        51
  mexican_style             50
  british_style             44
  indian_style              43
  exclude_beans             41
  spanish_style             40
  japanese_style            39
  exclude_yogurt            37
  exclude_egg               36
  american_style            35
  sweet_style               34
  exclude_mushroom          33

Rejection-mode distribution (in y'):
  constraint_violation 405
  unsafe_equipment     400
  too_many_steps       399
  extra_ingre

## 9. Format the dataset for each fine-tuning method

Each Azure fine-tuning method expects a different JSONL schema. See the relevant Azure OpenAI fine-tuning docs for the exact format; the functions below follow those conventions.

In [ ]:
def format_prompt(ingredients: list[str], user_constraint: str = "") -> str:
    """The user-facing prompt. Same format at train and eval time."""
    constraint_phrase = f"{user_constraint} " if user_constraint else ""
    return (
        f"Create a quick, student-friendly {constraint_phrase}recipe using these ingredients: "
        f"{', '.join(ingredients)}. "
        f"Keep it under {CFG.max_cook_minutes} minutes with {CFG.min_steps}-{CFG.max_steps} clear steps."
    )

def format_recipe_output(recipe: dict) -> str:
    """Render a recipe dict into the plain-text format we want the fine-tuned model to learn."""
    steps = "\n".join(f"{i+1}. {step}" for i, step in enumerate(recipe["steps"]))
    return f"**{recipe['title']}** ({recipe['total_time_minutes']} min)\n\n{steps}"

SYSTEM_PROMPT = (
    f"You are a recipe assistant for college students. "
    f"Generate quick recipes (under {CFG.max_cook_minutes} minutes) with "
    f"{CFG.min_steps}-{CFG.max_steps} clear numbered steps, using only the ingredients provided."
)


In [ ]:
def sample_metadata(sample: dict) -> dict:
    return {
        "constraint_category": sample.get("constraint_category", ""),
        "constraint": sample.get("constraint", ""),
        "user_constraint": sample.get("user_constraint", ""),
    }

def to_sft_record(sample: dict) -> dict:
    """SFT format: messages with the full assistant response.

    Note: Azure SFT generally expects each line to be just a messages object.
    We therefore do NOT add metadata here. The metadata remains in pool.jsonl.
    """
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": format_prompt(sample["ingredients"], sample.get("user_constraint", ""))},
            {"role": "assistant", "content": format_recipe_output(sample["preferred"])},
        ]
    }

def to_dpo_record(sample: dict) -> dict:
    """DPO format: input messages + preferred/non-preferred completions.

    Note: keep schema clean for Azure DPO; metadata remains in pool.jsonl.
    """
    return {
        "input": {
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": format_prompt(sample["ingredients"], sample.get("user_constraint", ""))},
            ]
        },
        "preferred_output": [
            {"role": "assistant", "content": format_recipe_output(sample["preferred"])}
        ],
        "non_preferred_output": [
            {"role": "assistant", "content": format_recipe_output(sample["rejected"])}
        ],
    }


In [ ]:
def write_jsonl(records: list[dict], path: str):
    with open(path, "w") as f:
        for r in records:
            f.write(json.dumps(r) + "\n")
    print(f"  wrote {len(records):5d} records -> {path}")

In [ ]:
def split_and_write(pool: list[dict], cfg: Config):
    REQUIRED = 1600 + 400

    if len(pool) < REQUIRED:
        raise ValueError(f"Need at least {REQUIRED} samples, got {len(pool)}")

    # Deterministic shuffle
    split_rng = random.Random(cfg.seed + 1)
    shuffled = pool.copy()
    split_rng.shuffle(shuffled)

    train = shuffled[:1600]
    val   = shuffled[1600:2000]

    print(f"Split: {len(train)} train / {len(val)} val ")

    out = cfg.out_dir

    # -------- SFT --------
    print("\nSFT:")
    write_jsonl([to_sft_record(s) for s in train], f"{out}/sft_train.jsonl")
    write_jsonl([to_sft_record(s) for s in val],   f"{out}/sft_val.jsonl")

    # -------- DPO --------
    print("\nDPO:")
    write_jsonl([to_dpo_record(s) for s in train], f"{out}/dpo_train.jsonl")
    write_jsonl([to_dpo_record(s) for s in val],   f"{out}/dpo_val.jsonl")

In [ ]:
split_and_write(pool, CFG)

Split: 1600 train / 400 val 

SFT:
  wrote  1600 records -> data_constrained/sft_train.jsonl
  wrote   400 records -> data_constrained/sft_val.jsonl

DPO:
  wrote  1600 records -> data_constrained/dpo_train.jsonl
  wrote   400 records -> data_constrained/dpo_val.jsonl


## 10. Spot-check the output files

One sample record from each file, to make sure the schema matches what the Azure fine-tuning job expects. If any of these look off, fix the formatter before launching a training job (fine-tuning failures for format reasons are slow to diagnose and cost credit).

In [ ]:
for name in ["sft_train", "dpo_train"]:
    path = f"{CFG.out_dir}/{name}.jsonl"
    print(f"\n=== {path} ===")
    with open(path) as f:
        first = json.loads(f.readline())
    print(json.dumps(first, indent=2)[:1200])

## 11. Next steps — hook into the Azure starter notebook

The files above drop directly into the fine-tuning cells of `cis5270_azure_example.ipynb`:

```python
# SFT
training_file_path   = "data/sft_train.jsonl"
validation_file_path = "data/sft_val.jsonl"

# DPO
training_file_path   = "data/dpo_train.jsonl"
validation_file_path = "data/dpo_val.jsonl"
```

For RFT, the starter notebook's grader is a simple `string_check` / `exact_match`. That's a weak signal for recipes — consider swapping it for a multi-component grader that matches the proposal's reward function (ingredient compliance + simplicity + structure + LLM-as-judge). Azure's fine-tuning API supports `multi_grader` and `score_model` grader types; the RFT records already include `ingredients`, `min_steps`, `max_steps`, and `max_time` fields to make that easier.

### Known failure modes to watch for on a bigger run

- **Acceptance rate too low** (< 60%): the judge prompt is too strict or the preferred-recipe prompt is too loose. Check the `verdict.reason` field on a sample of rejected candidates.
- **Rejected recipes too obviously bad**: DPO will overfit on superficial cues (e.g., length). If `_rejection_mode` is overwhelmingly `too_long`, the model learns "longer = worse" instead of the actual constraints. Inspect the mode distribution and rebalance `REJECTION_MODES` weights if needed.
- **Ingredient lists too homogeneous**: if the cuisine distribution skews, add more seeds or raise the temperature on call 1.
- **JSON parse failures**: `llm_json` returns `None` on parse errors. If the failure count in the console output is high, consider tightening the JSON instruction or adding a fallback regex-extract step.